#### 0. Install dependencies

In [1]:
# pip install -r requirements.txt

#### 1. Import essential tools and set up OpenAI's API environment.

In [2]:
import os
from pathlib import Path

from openai import AzureOpenAI
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import chromadb
import gradio as gr

#Remember to create a .env file in this environment in a folder location
load_dotenv(".env")
#load_dotenv()

# Initialize client once
client = AzureOpenAI(
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT")
)

chat_depolyment = os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT")
embedding_deployment = os.getenv('AZURE_OPENAI_EMBEDDING_DEPLOYMENT')

In [3]:
BASE_DIR = Path.cwd()
PDF_PATH = BASE_DIR / 'data' / '1728286846_the_nestle_hr_policy_pdf_2012.pdf'
CHROMA_DIR = BASE_DIR / 'chroma_db'

#### 2 Load Nestle's HR policy using PyPDFLoader and split it for easy processing.

In [4]:
loader = PyPDFLoader(str(PDF_PATH))
documents = loader.load()

# len(documents), documents[0].metadata
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=512,
    chunk_overlap=64,
)

chunks = text_splitter.split_documents(documents)
len(chunks)

35

#### 3 Create vector representations for text chunks using Chroma dB and OpenAI's embeddings.

In [5]:
def embed_texts(texts):
    response = client.embeddings.create(
        model=embedding_deployment, 
        input=texts)
    return [item.embedding for item in response.data]

def embed_query(text):
    response = client.embeddings.create(
        model=embedding_deployment, 
        input=text)
    return response.data[0].embedding

chunk_texts = [chunk.page_content for chunk in chunks]
chunk_metadatas = [chunk.metadata for chunk in chunks]
chunk_ids = [f'chunk-{index}' for index in range(len(chunks))]
chunk_embeddings = embed_texts(chunk_texts)

chroma_client = chromadb.PersistentClient(path=str(CHROMA_DIR))
collection = chroma_client.get_or_create_collection(name='nestle_hr_policy')
existing_ids = collection.get()['ids']
if existing_ids:
    collection.delete(ids=existing_ids)

collection.add(
    ids=chunk_ids,
    documents=chunk_texts,
    metadatas=chunk_metadatas,
    embeddings=chunk_embeddings,
)

#### 4. Create a prompt template to guide the chatbot in understanding and responding to users.

In [6]:
SYSTEM_PROMPT = '''
You are an intelligent HR assistant for Nestlé, designed to help employees 
and managers navigate company HR policies and reports accurately.

You will be provided with relevant excerpts retrieved from Nestlé's official 
HR policy documents, each labeled with a Source number and page number in 
the format: "Source 1 | page 3"

Instructions:
- Answer using ONLY the information provided in the context.
- Do not infer, assume, or supplement with knowledge outside the provided context.
- Always cite your answer using the Source number and page (e.g., Source 1, page 3).
- If multiple sources are relevant, cite all of them.
- Maintain a professional, friendly, and helpful tone in all responses.
- If the user's question is a follow-up, take into account the prior conversation 
  history to provide a coherent and consistent response.
- If the context does not contain sufficient information to answer the question, 
  respond exactly with:
  "The provided policy document does not contain sufficient information to answer 
  this question. Please contact Nestlé HR directly for further assistance."

Response format:
**Answer**: [Your clear and concise response here]
**Source**: [e.g., Source 1 (page 3), Source 2 (page 7)]
'''

#### 5. Build a question-answering system using the GPT-3.5 Turbo model to retrieve answers from text chunks.

In [ ]:
def retrieve_docs(question, k=4):
    results = collection.query(
        query_embeddings=[embed_query(question)],
        n_results=k
    )

    docs = []
    for document, metadata in zip(results['documents'][0], results['metadatas'][0]):
        docs.append({'page_content': document, 'metadata': metadata})
    return docs

def format_context(retrieved_docs):
    parts = []
    for index, doc in enumerate(retrieved_docs, start=1):
        # if metadata has page return page number otherwise return unknown
        page = doc['metadata'].get('page', 'unknown')
        parts.append(f"Source {index} | page {page}\n{doc['page_content']}")
    return '\n\n'.join(parts)

def answer_question(question):
    retrieved_docs = retrieve_docs(question)
    context = format_context(retrieved_docs)
    response = client.chat.completions.create(
        model=chat_depolyment,
        temperature=0,
        messages=[
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': f'''
                Context:\n{context}\n\n
                Question:\n{question}''',
            },
        ],
    )
    return response.choices[0].message.content

In [8]:
answer_question('What does the policy say about Employee relations?')
# user_question = input('Enter your HR policy question: ')
# answer_question(user_question)

'**Answer**: The policy on Employee Relations emphasizes that Nestlé\'s Human Resources management supports an organization that is dynamic and evolving ("on the move"). Nestlé is committed to establishing flat and flexible organizational structures with minimal management layers and broad spans of control. This approach is intended to enable people development, increase efficiency, and facilitate the implementation of Nestlé\'s Management and Leadership Principles. Additionally, Nestlé fosters a culture based on trust, mutual respect, and dialogue, where management and employees work daily to create and maintain positive individual and collective relationships. The company also upholds the freedom of association and the right to collective bargaining for its employees. HR ensures a respectful dialogue and that employees\' voices are heard, sustaining an environment of mutual trust within teams (Source 1, page 4; Source 2, page 6; Source 3, page 6). \n\n**Source**: Source 1 (page 4), S

#### 6. Use Gradio to build a user-friendly chatbot interface, enabling interaction and information retrieval.

In [9]:
import gradio as gr

def process_user_input(user_input):
    if not user_input or not user_input.strip():
        return 'Please enter a question about the HR policy.'
    if 'answer_question' not in globals():
        return 'Please run the previous notebook cells first so the retrieval and chat functions are loaded.'
    return answer_question(user_input)

def gradio_chat(message, history):
    history = history or []
    if not message.strip():
        return '', history

    answer = process_user_input(message)
    history = history + [
        {'role': 'user', 'content': message},
        {'role': 'assistant', 'content': answer},
    ]
    return '', history

with gr.Blocks(title='Nestle HR Assistant') as demo:
    gr.Markdown('# Nestle HR Policy Assistant')
    gr.Markdown('Ask questions about the Nestle HR policy document.')

    chatbot = gr.Chatbot(label='Conversation', height=450, type='messages')
    user_input = gr.Textbox(
        label='Your question',
        placeholder='Example: What does the policy say about Employee relations?'
    )
    clear_button = gr.Button('Clear Chat')

    user_input.submit(gradio_chat, inputs=[user_input, chatbot], outputs=[user_input, chatbot])
    clear_button.click(lambda: ('', []), outputs=[user_input, chatbot])

demo.launch()

TypeError: Chatbot.__init__() got an unexpected keyword argument 'type'